# 03 · Hard-negative mining  *(stage 5c)* — the best resultRe-trains with negatives mined from each anchor's **own nearest neighbours**instead of random ones. **~8 minutes on a T4.**Result: recall@10 **0.3214 → 0.3393**. Against the untuned baseline:0.2778 → 0.3393, **+22%**, McNemar exact **p = 0.0010**.### WhyA user reported a false positive: a *feature request* ("allow configuring thedefault Changes view changeset") matched a *bug report* ("Changes view breakswhen selecting Last Turn's Changes"). Same feature area, shared phrase, notduplicates.The cause was the training objective. MNRL's in-batch negatives are otherduplicate pairs — almost always about entirely unrelated features. Triviallyeasy. The model was never asked to separate *"same area, different intent"*.### Upload| file | produced by ||---|---|| `triplets.jsonl.gz` | `python -m src.finetune --mine-negatives colab/triplets.jsonl.gz` || `texts_clean.jsonl.gz` | `python -m src.embed --export-texts ... --clean` |### How the negatives are chosen (`src/finetune.py`)- **Ranks 5–60**, not 1–5. The very nearest neighbours are often *real*  duplicates nobody linked; training against those teaches the opposite of the  goal.- **False-negative guard.** If A and B are both duplicates of C, then A and B are  duplicates of *each other* — so B can never be a negative for A. The whole  duplicate chain is excluded, and **text** is compared as well as issue number,  because different numbers sometimes carry identical text.

In [ ]:
!pip install -q sentence-transformers huggingface_hub

In [ ]:
import os# Recommended by the OOM error itself; reduces allocator fragmentation.os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"import torch, gcgc.collect(); torch.cuda.empty_cache()assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"print(torch.cuda.get_device_name(0),      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

### Two settings that exist because this OOMed**`max_seq_length = 256`.** Training text is p50 **150** tokens, p90 480. The512 default pads every batch to its longest member, and attention cost isquadratic in length. 256 covers 72% of texts whole and cuts memory ~4×.**`batch_size = 16`, not 32.** Triplets feed *three* texts per example, so abatch of 32 is 96 sequences per step. The usual objection — smaller batches meanfewer in-batch negatives — matters much less here, because every example nowcarries **3 explicit hard negatives**, which are far better than random ones.If it still OOMs: `Runtime > Disconnect and delete runtime` for a fresh GPU."Restart session" keeps files but does not always release GPU memory held by aprevious run.

In [ ]:
import gzip, json, random, numpy as np, torchfrom sentence_transformers import InputExample, SentenceTransformer, lossesfrom torch.utils.data import DataLoaderrandom.seed(0); np.random.seed(0); torch.manual_seed(0); torch.cuda.manual_seed_all(0)rows = [json.loads(l) for l in gzip.open("triplets.jsonl.gz", "rt")]random.Random(0).shuffle(rows)print(f"{len(rows):,} triplets (anchor, positive, hard negative)")model = SentenceTransformer("BAAI/bge-small-en-v1.5")model.max_seq_length = 256loader = DataLoader([InputExample(texts=[r["a"], r["p"], r["n"]]) for r in rows],                    shuffle=True, batch_size=16, drop_last=True)loss = losses.MultipleNegativesRankingLoss(model)# 2 epochs, not 3: 6,000 triplets is 3x the data of notebook 02, so the same# number of gradient steps arrives sooner.model.fit(train_objectives=[(loader, loss)], epochs=2,          warmup_steps=int(len(loader) * 2 * 0.1),          optimizer_params={"lr": 2e-5},          output_path="ft-hardneg", show_progress_bar=True)print("peak GPU:", torch.cuda.max_memory_allocated() / 1e9, "GB")

In [ ]:
from huggingface_hub import login, HfApilogin()REPO = "Musab6969/bge-small-vscode-dup-hardneg"api = HfApi()api.create_repo(REPO, repo_type="model", private=False, exist_ok=True)model.save("ft-hardneg")api.upload_folder(folder_path="ft-hardneg", repo_id=REPO, repo_type="model",                  commit_message="hard-negative mined fine-tune")# max_seq_length must persist: the Space embeds queries with this model, and a# mismatch against how the corpus was encoded degrades retrieval silently.print(json.load(open("ft-hardneg/sentence_bert_config.json")))

In [ ]:
import gcgc.collect(); torch.cuda.empty_cache()from pathlib import Pathrecs = [json.loads(l) for l in gzip.open("texts_clean.jsonl.gz", "rt")]Path("shards_hn").mkdir(exist_ok=True)for s in range(0, len(recs), 10_000):    chunk = recs[s:s + 10_000]    v = model.encode([r["t"] for r in chunk], batch_size=64,                     normalize_embeddings=True, show_progress_bar=True,                     convert_to_numpy=True).astype(np.float32)    i = s // 10_000    np.save(f"shards_hn/emb_{i:05d}.npy", v)    np.save(f"shards_hn/ids_{i:05d}.npy", np.array([r["n"] for r in chunk], dtype=np.int64))    print("shard", i)!zip -qr shards_hn.zip shards_hnfrom google.colab import filesfiles.download("shards_hn.zip")

### Back on the laptop```bashunzip shards_hn.zippython -m src.embed --import-vectors shards_hn/ --model-key bge-small-ft-hardnegpython -m src.retrievepython scripts/export.py          # rebuild serving artifacts```